# NewsLens — Bias Classifier Comparison (RQ2)

Fine-tunes and compares `bert-base-uncased`, `roberta-base`, and
`distilbert-base-uncased` as sentence-level bias classifiers on BASIL,
answering RQ2: *which transformer backbone best distinguishes biased from
neutral sentences?*

This notebook is **self-contained** — it duplicates the labeling/training
logic from `analysis/bias_classifier.py` in the main repo so it runs on
Kaggle without needing that repo importable. If you'd rather import the
real module directly: zip the repo, upload it as a private Kaggle Dataset
via *Add Data → New Dataset*, attach it here, and `sys.path.append` its
path instead of re-defining everything below.

## Before running

1. **Settings (right sidebar) → Accelerator → GPU T4 x2** (or P100/whatever's available).
2. **Settings → Internet → On** — needed to download the pretrained
   checkpoints from Hugging Face and to `pip install` a couple of packages.
3. **Add Data → Upload/attach your BASIL dataset.** Arrange it beforehand as
   `<event_id>/<source>.json` per event, zip it, upload as a Kaggle Dataset,
   then attach it to this notebook. After attaching, run
   `!ls /kaggle/input` to see the mounted path and update `BASIL_DIR` in the
   config cell below to match.
4. Expect the full 3-checkpoint sweep to take roughly 1–3 hours on a T4,
   depending on BASIL's actual size once you've downloaded it — matches the
   spec's "a few hours on a free GPU tier" estimate, just on Kaggle's GPU
   instead of Colab's.


In [ ]:
# Extra packages — Kaggle's base image usually has transformers/torch/scikit-learn
# already, but can lag a version behind. Safe to re-run even if already present.
!pip install -q -U datasets accelerate
!python -m spacy download en_core_web_sm -q


In [ ]:
import json
import time
import glob
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
import torch
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ---- CONFIG — edit these two paths for your setup ----

# Run `!ls /kaggle/input` after attaching your BASIL dataset and paste the
# resulting folder name here.
BASIL_DIR = Path("/kaggle/input/basil-dataset")   # <-- EDIT ME

OUTPUT_ROOT = Path("/kaggle/working/bias_classifier_runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CHECKPOINTS = ["bert-base-uncased", "roberta-base", "distilbert-base-uncased"]
EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
MAX_LENGTH = 128
SEED = 42

!ls /kaggle/input


## BASIL loading

Same caveat as the main repo's `ingestion/basil_loader.py`: this assumes the
commonly-used layout (`body-paragraphs`, `word-level-annotations`). **Open
one of your actual downloaded JSON files and check these two key names
before trusting this cell** — adjust `_extract_text`/`_extract_bias_spans`
if your copy differs.


In [ ]:
@dataclass
class BiasSpan:
    start: int
    end: int
    label: str
    text: str

@dataclass
class BasilArticle:
    event_id: str
    source: str
    raw_text: str
    url: str | None
    bias_spans: list = field(default_factory=list)

def _extract_text(article_json: dict) -> str:
    paragraphs = article_json.get("body-paragraphs") or article_json.get("sentences") or []
    sentences = []
    for para in paragraphs:
        if isinstance(para, list):
            sentences.extend(para)
        else:
            sentences.append(para)
    return " ".join(s.strip() for s in sentences if s and s.strip())

def _extract_bias_spans(article_json: dict) -> list:
    raw_spans = article_json.get("word-level-annotations") or article_json.get("annotations") or []
    spans = []
    for s in raw_spans:
        try:
            spans.append(BiasSpan(
                start=s.get("start", -1),
                end=s.get("end", -1),
                label=s.get("bias", s.get("label", "unknown")),
                text=s.get("text", ""),
            ))
        except AttributeError:
            continue
    return spans

def list_event_ids(basil_dir: Path) -> list:
    if not basil_dir.exists():
        raise FileNotFoundError(f"BASIL_DIR '{basil_dir}' not found — check the attached dataset path.")
    return sorted(p.name for p in basil_dir.iterdir() if p.is_dir())

def load_event(event_id: str, basil_dir: Path) -> list:
    event_dir = basil_dir / event_id
    articles = []
    for json_path in sorted(event_dir.glob("*.json")):
        source = json_path.stem
        with open(json_path, "r", encoding="utf-8") as f:
            article_json = json.load(f)
        articles.append(BasilArticle(
            event_id=event_id,
            source=source,
            raw_text=_extract_text(article_json),
            url=article_json.get("url"),
            bias_spans=_extract_bias_spans(article_json),
        ))
    return articles

def load_all_events(basil_dir: Path) -> dict:
    return {eid: load_event(eid, basil_dir) for eid in list_event_ids(basil_dir)}

events = load_all_events(BASIL_DIR)
print(f"Loaded {len(events)} events, {sum(len(a) for a in events.values())} articles total")


## Labeling

A sentence is labeled biased (1) if the *text* of any annotated bias span is
a substring of it, case-insensitive — more robust than trusting BASIL's raw
character offsets, which depend on paragraph-joining exactly matching the
original file's format (unverified, see the loading cell above).


In [ ]:
nlp = spacy.load("en_core_web_sm")

def to_sentences(text: str) -> list:
    return [s.text.strip() for s in nlp(text).sents if s.text.strip()]

@dataclass
class LabeledSentence:
    event_id: str
    source: str
    article_key: str
    sentence: str
    label: int

def label_article(article: BasilArticle) -> list:
    sentences = to_sentences(article.raw_text)
    span_texts = [s.text.strip().lower() for s in article.bias_spans if s.text.strip()]
    article_key = f"{article.event_id}:{article.source}"
    rows = []
    for sent in sentences:
        sent_lower = sent.lower()
        label = int(any(t in sent_lower for t in span_texts))
        rows.append(LabeledSentence(article.event_id, article.source, article_key, sent, label))
    return rows

def build_labeled_dataset(events: dict) -> list:
    rows = []
    for articles in events.values():
        for article in articles:
            rows.extend(label_article(article))
    return rows

rows = build_labeled_dataset(events)
print(f"{len(rows)} labeled sentences, {sum(r.label for r in rows)} positive "
      f"({sum(r.label for r in rows) / len(rows):.1%})")


## Article-level train/test split

Split by **article**, not by sentence — sentences from the same article are
too similar to each other (shared topic, shared authorial voice); a
sentence-level split leaks information between train/test and inflates
reported accuracy.


In [ ]:
def split_by_article(rows: list, test_size: float = 0.2, seed: int = SEED):
    groups = [r.article_key for r in rows]
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(splitter.split(rows, groups=groups))
    return [rows[i] for i in train_idx], [rows[i] for i in test_idx]

def class_balance(rows: list) -> dict:
    labels = [r.label for r in rows]
    n = len(labels)
    return {"n": n, "positive": sum(labels), "positive_frac": sum(labels) / n if n else 0.0}

train_rows, test_rows = split_by_article(rows)
print("Train:", class_balance(train_rows))
print("Test: ", class_balance(test_rows))


## Train + evaluate one checkpoint

Runs for each of the three checkpoints in the next cell. Inference time is
timed separately from `Trainer.evaluate()` (which includes loss computation
overhead not representative of deployed single-sentence inference).


In [ ]:
def train_and_evaluate(checkpoint: str, train_rows: list, test_rows: list, output_dir: str):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

    def to_hf_dataset(rows):
        return Dataset.from_dict({"text": [r.sentence for r in rows], "label": [r.label for r in rows]})

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    train_ds = to_hf_dataset(train_rows).map(tokenize, batched=True)
    test_ds = to_hf_dataset(test_rows).map(tokenize, batched=True)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds, zero_division=0)}

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        report_to=[],
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
                       compute_metrics=compute_metrics)
    trainer.train()
    eval_metrics = trainer.evaluate()

    model.eval()
    device = next(model.parameters()).device
    sample = test_rows[: min(50, len(test_rows))]
    inputs = tokenizer([r.sentence for r in sample], truncation=True, padding=True,
                        max_length=MAX_LENGTH, return_tensors="pt").to(device)
    with torch.no_grad():
        start = time.perf_counter()
        model(**inputs)
        elapsed = time.perf_counter() - start
    inference_ms = (elapsed / len(sample)) * 1000

    metrics = {
        "checkpoint": checkpoint,
        "accuracy": eval_metrics["eval_accuracy"],
        "f1": eval_metrics["eval_f1"],
        "inference_ms_per_sentence": inference_ms,
        "train_n": len(train_rows),
        "test_n": len(test_rows),
    }
    return metrics, model, tokenizer


In [ ]:
results = []

for checkpoint in CHECKPOINTS:
    print(f"\n=== Training {checkpoint} ===")
    metrics, model, tokenizer = train_and_evaluate(
        checkpoint, train_rows, test_rows, output_dir=str(OUTPUT_ROOT / checkpoint)
    )
    results.append(metrics)

    save_path = OUTPUT_ROOT / checkpoint / "final"
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    del model, tokenizer
    torch.cuda.empty_cache()

    # Trainer writes intermediate checkpoint-* folders under output_dir even with
    # save_strategy="no" turned off for the *final* save; clear them to keep
    # /kaggle/working under Kaggle's output size limit before moving to the next model.
    for stray in glob.glob(str(OUTPUT_ROOT / checkpoint / "checkpoint-*")):
        import shutil
        shutil.rmtree(stray, ignore_errors=True)


In [ ]:
results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
results_df.to_csv("/kaggle/working/bias_classifier_results.csv", index=False)
results_df


## Next steps

- **Commit this notebook** ("Save Version → Save & Run All") — the files
  under `/kaggle/working/` (the results CSV and each checkpoint's `final/`
  weights folder) then appear in the notebook's **Output** tab for download.
- Model folders are ~250–500MB each (base BERT/RoBERTa) or ~250MB
  (DistilBERT) — if you hit Kaggle's output size limit, delete the
  non-winning checkpoints' `final/` folders before committing and keep only
  the best one plus the results CSV.
- Whichever checkpoint wins on F1 (BASIL's bias labels are imbalanced —
  most sentences aren't flagged as biased — so F1 matters more than raw
  accuracy here) is what section 6.7 in the main spec expects you to report
  for RQ2, alongside accuracy and inference time as a secondary
  deployment-cost consideration.
- To wire the winning model back into the NewsLens API: download its
  `final/` folder, drop it into the main repo (e.g.
  `newslens/models/bias_classifier/`), and load it in
  `analysis/bias_classifier.py` with
  `AutoModelForSequenceClassification.from_pretrained("./models/bias_classifier")`
  instead of a Hub checkpoint name.
